# Attempting to Model NGC6569 with PyfalcON


In [1]:
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation

import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 200  # Set to 100MB or whatever you need

from IPython.display import HTML

import pandas as pd

import pyfalcon

In [2]:
import astropy.coordinates as coord
import astropy.units as u
from astropy.constants import G
import gala.coordinates as gc
import gala.dynamics as gd
import gala.potential as gp
from gala.units import galactic
from gala.dynamics import mockstream as ms

import agama

import importlib
import sys
from pathlib import Path

from time import time

START_DIR = Path.cwd().resolve()
NGC6569_DIR = next(
    (path for path in (START_DIR, START_DIR / "ngc6569", START_DIR / "joe's_Code" / "ngc6569")
     if (path / "milkyway").is_dir()),
    None,
)
if NGC6569_DIR is None:
    raise FileNotFoundError("Could not locate joe's_Code/ngc6569/milkyway from the current working directory")

DATA_DIR = NGC6569_DIR / "data"
OUTPUT_DIR = NGC6569_DIR / "output"
MILKYWAY_DIR = NGC6569_DIR / "milkyway"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(NGC6569_DIR) not in sys.path:
    sys.path.insert(0, str(NGC6569_DIR))

from leap_frog import kdk_leapfrog_TD
from leap_frog import kdk_leapfrog

#from leap_frog_hdf5 import kdk_leapfrog_hdf5, load_simulation_hdf5

In [3]:
# default Astropy Galactocentric frame parameters to the values adopted in Astropy v4.0:
_ = coord.galactocentric_frame_defaults.set('v4.0')

# set Agama units 
# working units: 1 Msun, 1 kpc, 1 km/s
agama.setUnits(length=1*u.kpc, velocity=1*u.km/u.s, mass=1*u.Msun)
print("Newton G in Agama units,",agama.G)

# Check the current unit system
print("Current Agama units:")
print(f"Length unit: {agama.getUnits()['length']}")
print(f"Velocity unit: {agama.getUnits()['velocity']}")  
print(f"Time unit: {agama.getUnits()['time']}")
print(f"Mass unit: {agama.getUnits()['mass']}")

agama_time_unit = agama.getUnits()["time"]
print(agama_time_unit)

Newton G in Agama units, 4.30091727067736e-06
Current Agama units:
Length unit: 1.0 kpc
Velocity unit: 1.0 km / s
Time unit: 977.792221683525 Myr
Mass unit: 1.0 solMass
977.792221683525 Myr


In [4]:
# Use the Hunter rotating potential 
pot_ext = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_full.ini")) 
pot_rot = agama.Potential(str(MILKYWAY_DIR / "MWPotentialHunter24_rotating.ini")) 
pot_bovy = agama.Potential(str(MILKYWAY_DIR / "MWPotential2014.ini")) 

pot_use = pot_rot

In [6]:
# NG6569 coordinates 

c = coord.SkyCoord(ra = 273.412*u.degree, dec = -31.827*u.degree,
                        distance=(10.5)*u.kpc,
                        pm_ra_cosdec= -4.125*u.mas/u.yr,
                        pm_dec= -7.354*u.mas/u.yr,
                        radial_velocity= -49.82*u.km/u.s)

# transform to galactic centeric 
c_gc = c.transform_to(coord.Galactocentric).data
print(c_gc._differentials)

{'s': <CartesianDifferential (d_x, d_y, d_z) in kpc mas / (rad yr)
    (-6.71192364, -36.78139749, 5.0482508)>}


In [7]:
# creat phase space object
w0 = gd.PhaseSpacePosition(c_gc)
print("initial position :", w0.pos)
print("initial velocity :", w0.vel)
print("Need to convert velocity to km/s")

pos_0 = np.r_[w0.pos.x.value, w0.pos.y.value, w0.pos.z.value]
vel_0 = np.r_[w0.vel.d_x.to(u.km/u.s).value, 
              w0.vel.d_y.to(u.km/u.s).value,
              w0.vel.d_z.to(u.km/u.s).value]
print("position", pos_0) 
print("velocity", vel_0) 

initial position : (2.30319138, 0.08753024, -1.22751023) kpc
initial velocity : (-6.71192364, -36.78139749, 5.0482508) kpc mas / (rad yr)
Need to convert velocity to km/s
position [ 2.30319138  0.08753024 -1.22751023]
velocity [ -31.81767579 -174.36112841   23.93108381]


In [ ]:
# integrate orbit 
tfin= -1000*u.Myr
nt=2000
t_eval = np.linspace(0, tfin, nt)
t_scipy = t_eval.to(u.Gyr).value/agama_time_unit.to(u.Gyr).value
t_scipy

In [11]:
def rhs(t,state): 
    
    pos = state[:3]
    vel = state[3:]

    acc = pot_use.force(pos, t=t)
    return np.r_[vel, acc] 

# integrate with solve_ivp
from scipy.integrate import solve_ivp

state_0 = np.r_[pos_0, vel_0] 
print(state_0, state_0.shape)

t_span = (0, t_scipy[-1]) 
sol = solve_ivp(rhs, t_span, state_0, t_eval =t_scipy, rtol=1e-10, atol=1e-10)
orbit = sol["y"]

[ 2.30319138e+00  8.75302429e-02 -1.22751023e+00 -3.18176758e+01
 -1.74361128e+02  2.39310838e+01] (6,)


In [12]:
x = orbit[0,:]
y = orbit[1,:]
z = orbit[2,:]
vx = orbit[3,:]
vy = orbit[4,:]
vz = orbit[5,:]

In [18]:
# Define the parameters for your King model
W0_value = 7.0  # Example W0 value
# create an isolated star cluster
r_scale = 1/1000
m = 2.3*1e5*(2)
pot_sat = agama.Potential(type='king', W0=W0_value, scaleRadius=r_scale, mass=m)
df_sat = agama.DistributionFunction(type='quasispherical', potential=pot_sat)
Nbody = 150000
xv, mass = agama.GalaxyModel(pot_sat, df_sat).sample(Nbody)

r_agama = np.sqrt(xv[:,0]**2 + xv[:,1]**2 + xv[:,2]**2) 
v_agama = np.sqrt(xv[:,3]**2 + xv[:,4]**2 + xv[:,5]**2) 

print("Agama G:", agama.G)

cluster_data = np.c_[mass, xv]
cluster_data.shape
np.savetxt(DATA_DIR / "cluster_data.txt", cluster_data)

Agama G: 4.30091727067736e-06


In [ ]:
pos_0 = np.r_[x[-1], y[-1], z[-1]]
vel_0 = np.r_[vx[-1], vy[-1], vz[-1]]
print(pos_0)
print(vel_0) 

In [288]:
print("make this a function")

def shift_to_gc(xv, pos_0, vel_0): 

    x = xv[:,0] + pos_0[0]
    y = xv[:,1] + pos_0[1]
    z = xv[:,2] + pos_0[2]

    vx = xv[:,3] + vel_0[0]
    vy = xv[:,4] + vel_0[1]
    vz = xv[:,5] + vel_0[2]


    print("checks")
    print(np.mean(x), np.mean(y), np.mean(z))
    print(pos_0)


    print(np.mean(vx), np.mean(vy), np.mean(vz))
    print(vel_0)

    return np.column_stack((x, y, z)), np.column_stack((vx, vy, vz))


make this a function


In [289]:
# pos_0 = np.column_stack((x, y, z))
# vel_0 = np.column_stack((vx, vy, vz))
# vel_0.shape

pos_0 , vel_0 = shift_to_gc(xv, pos_0, vel_0)

checks
-0.13355979463122905 -2.27827702596787 -0.4425933187302422
[-0.13356328 -2.27827074 -0.44259326]
-129.14879461766608 -82.12625463687772 -0.4449089584146095
[-129.16134153  -82.14009399   -0.44259326]


In [291]:
time_unit=u.kpc.to(u.km)*u.s.to(u.Gyr)
time_unit

0.9777922216807892

In [292]:
tmax = -tfin.to(u.Gyr).value/time_unit
print("maximum time", tmax)
print(t_scipy[-1])

maximum time 0.204542433009139
-0.2045424330085667


In [293]:
kmax=16
tau = 2**(-kmax)*time_unit
print("time step:", tau) 
nt=int(tmax/tau) + 1
print("number of time steps:", nt)

eps = 1/1000  # II 
eps = .1/1000 # I 
eps_power = -4
eps = (2**eps_power)/1000
print("softening length:", eps) 

time step: 1.4919925257580401e-05
number of time steps: 13710
softening length: 6.25e-05


In [296]:
downsample=20
#filename="ngc_6569_runI"
#nt=1000

In [299]:
t1 = time()
sim_data = kdk_leapfrog(pot_use, pos_0, vel_0, 
                        mass, nt, tau, agama.G, eps, 
                        time_unit, downsample,last_snapshot=False)
t2=time()
print("run time", t2-t1, (t2-t1)/60)

run time 1616.840439081192 26.9473406513532


In [304]:
from bound_funcs import get_bound_particles

bound_data = get_bound_particles(sim_data, mass)

In [307]:
def trajectories(bound_data, sim_data): 
    """
    Extract time evolution trajectories of bound cluster properties from simulation data.
    
    This function processes the output from bound particle tracking to create clean
    time series arrays for analysis and plotting of cluster evolution.
    
    Parameters:
    -----------
    bound_data : list of dict
        List of dictionaries containing bound particle data at each timestep.
        Each dictionary should contain keys: 'total mass', 'total energy', 'pos', 'vel'
    sim_data : list of dict
        List of dictionaries containing simulation data at each timestep.
        Each dictionary should contain key: 'time'
        
    Returns:
    --------
    dict : Dictionary containing trajectory arrays
        'mass' : array, shape (n,) - Total mass of bound particles vs time
        'energy' : array, shape (n,) - Total energy of bound particles vs time  
        'time' : array, shape (n,) - Time array
        'pos' : array, shape (n, 3) - Center of mass position vs time
        'vel' : array, shape (n, 3) - Center of mass velocity vs time

    """
    
    n = len(bound_data)
    
    # Validate input lengths match
    if len(sim_data) != n:
        raise ValueError(f"Length mismatch: bound_data has {n} entries, sim_data has {len(sim_data)}")
    
    # Initialize trajectory arrays
    time_array = np.zeros(n)
    mass_traj = np.zeros(n)
    energy_traj = np.zeros(n)
    r_traj = np.zeros((n, 3))
    d_traj = np.zeros(n)
    v_traj = np.zeros((n, 3))
    
    # Extract data for each timestep
    for i in range(n): 
        # Extract bound particle properties
        mass_traj[i] = bound_data[i]["total mass"]
        energy_traj[i] = bound_data[i]["total energy"]
        r_traj[i, :] = bound_data[i]["pos"]
        v_traj[i, :] = bound_data[i]["vel"]
        d_traj[i] = np.linalg.norm(r_traj[i,:]) 
        
        # Extract time from simulation data
        time_array[i] = sim_data[i]["time"] 
    
    # Package results into dictionary
    traj = {
        'mass': mass_traj,
        'energy': energy_traj,
        'time': time_array,
        'pos': r_traj,
        'vel': v_traj,
        'distance': d_traj
    }
    
    return traj

traj = trajectories(bound_data, sim_data)

## Implemented: Dev3 Step 2 convergence sweep

The cells below add the cached one-parameter-at-a-time mass-loss worker and a gated driver with Dev3-centered sweep ranges. `RUN_NGC6569_STEP2` stays `False` by default so opening or running the notebook will not launch the expensive grid accidentally.

In [ ]:
## Step 2 cache helpers
# These helpers make each convergence run reproducible and reloadable from disk.
from pathlib import Path
import json
import hashlib

from gc_initial_conditions import sample_king_model  # King-model sampler used by each sweep point
from bound_funcs import get_bound_particles  # Bound-mass extractor used after each N-body run

# The Step 2 cells depend on variables created by the earlier notebook setup/run cells.
_required_step2_names = [
    "OUTPUT_DIR", "np", "u", "x", "y", "z", "vx", "vy", "vz",
    "m", "Nbody", "tfin", "downsample", "pot_use",
    "time_unit", "tmax", "kdk_leapfrog", "trajectories", "agama",
]
_missing_step2_names = [name for name in _required_step2_names if name not in globals()]
if _missing_step2_names:
    raise RuntimeError(
        "Run the earlier NGC6569 setup/simulation cells before Step 2. Missing: "
        + ", ".join(_missing_step2_names)
    )

# Keep this copy's Step 2 products separate from the 1 Gyr binary notebook cache.
STEP2_OUT = OUTPUT_DIR / "dev3_original_copy_step2"
STEP2_CACHE = STEP2_OUT / "cache"  # Per-run mass-loss curves live here.
STEP2_CACHE.mkdir(parents=True, exist_ok=True)

def write_manifest(path, manifest):
    # The JSON sidecar records enough metadata to reshape and interpret the raw binary table.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(manifest, indent=2))
    return path

def write_binary_table(path, data, columns, units=None, description=""):
    # Store numeric data as raw float64 and keep labels/units in a sidecar manifest.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(data, dtype=np.float64)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    arr.tofile(path)
    meta = {
        "binary_file": path.name,
        "dtype": "float64",
        "shape": list(arr.shape),
        "columns": list(columns),
        "units": units or {},
        "description": description,
    }
    write_manifest(path.with_suffix(".json"), meta)
    return path

def read_binary_table(path):
    # New caches have a JSON sidecar; old two-column caches can still be read without one.
    path = Path(path)
    meta_path = path.with_suffix(".json")
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        data = np.fromfile(path, dtype=np.dtype(meta["dtype"])).reshape(meta["shape"])
        return data, meta

    data = np.fromfile(path, dtype=np.float64)
    if data.size % 2 != 0:
        raise ValueError(f"Legacy cache {path} is not a two-column float64 table")
    return data.reshape((-1, 2)), {
        "binary_file": path.name,
        "dtype": "float64",
        "shape": [int(data.size // 2), 2],
        "columns": ["elapsed_time", "bound_mass_fraction"],
        "legacy_without_manifest": True,
    }

def binary_table_ready(path):
    return Path(path).exists()  # The worker treats an existing .bin as a cache hit.

def binary_cache_key(payload):
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:10]  # Stable compact key for filenames.

def king_trunc_ratio(W0, rgrid=None):
    """Return the approximate King-model truncation radius in scale-radius units."""
    if rgrid is None:
        rgrid = np.logspace(-4, 3, 4000)  # Dimensionless radius grid in units of scaleRadius.
    pot_king = agama.Potential(type="king", W0=W0, scaleRadius=1.0, mass=1.0)  # Unit model for W-only scaling.
    pts = np.column_stack((rgrid, np.zeros_like(rgrid), np.zeros_like(rgrid)))
    rho = pot_king.density(pts)  # Density falls sharply near the King truncation radius.
    return rgrid[rho > rho.max() * 1e-10].max()


In [ ]:
## Step 2 worker: one cached mass-loss simulation
# Each call either reloads one M(t)/M0 curve or runs one short N-body realization.
center_pos_0 = np.array([x[-1], y[-1], z[-1]], dtype=float)
center_vel_0 = np.array([vx[-1], vy[-1], vz[-1]], dtype=float)  # Use the true vz component for the orbit endpoint.

def xv_to_xyz_vel(xv, center_pos, center_vel):
    """Shift King-model phase-space offsets onto the NGC6569 orbit."""
    return xv[:, :3] + center_pos, xv[:, 3:] + center_vel

def _step2_cache_path(payload):
    h = binary_cache_key(payload)  # Hash all run parameters so different sweeps cannot collide.
    stem = (
        f"ml_W{payload['W0']:g}_r{payload['scale_radius_kpc']:.5f}_"
        f"k{payload['kmax']}_eps{payload['eps_pc']:g}_"
        f"m{payload['m_star']:g}_N{payload['Nbody']}_{h}"
    )
    return STEP2_CACHE / f"{stem}.bin"

def run_ngc6569_mass_loss(W0, scale_radius, kmax, eps_pc, m_star, use_cache=True):
    """Run or load one short NGC6569 mass-loss curve for Step 2 convergence tests."""
    if m_star <= 0:
        raise ValueError("m_star must be positive")

    Nbody_run = int(round(m / m_star))  # Heavier star mass means fewer, coarser particles.
    if Nbody_run < 1:
        raise ValueError(f"m_star={m_star:g} gives Nbody={Nbody_run}")

    payload = {  # Everything that changes the physical/numerical run belongs in the cache key.
        "W0": float(W0),
        "scale_radius_kpc": float(scale_radius),
        "kmax": int(kmax),
        "eps_pc": float(eps_pc),
        "m_star": float(m_star),
        "Nbody": int(Nbody_run),
        "cluster_mass_Msun": float(m),
        "tfin_Myr": float(tfin.to(u.Myr).value),
        "downsample": int(downsample),
        "potential": "MWPotentialHunter24_rotating.ini",
    }
    cache_path = _step2_cache_path(payload)
    if use_cache and binary_table_ready(cache_path):
        table, _ = read_binary_table(cache_path)  # Return cached t_Myr and M/M0 without re-simulating.
        print(f"cache hit: {cache_path.name}")
        return table[:, 0], table[:, 1]

    print(
        f"running W0={W0:g}, r_s={scale_radius * 1000.0:.4g} pc, "
        f"kmax={int(kmax)}, eps={eps_pc:g} pc, m_star={m_star:g} Msun, N={Nbody_run}"
    )
    xv_run, mass_run, _, _ = sample_king_model(W0, scale_radius, m, Nbody_run)  # Cluster frame.
    pos_run, vel_run = xv_to_xyz_vel(xv_run, center_pos_0, center_vel_0)  # Galactocentric frame.
    tau_run = 2 ** (-int(kmax)) * time_unit  # Convert kmax into the leapfrog timestep.
    nt_run = int(tmax / tau_run) + 1
    eps_run = eps_pc / 1000.0  # pc to kpc for the integrator.

    sim_run = kdk_leapfrog(
        pot_use, pos_run, vel_run, mass_run, nt_run, tau_run,
        agama.G, eps_run, time_unit, int(downsample), last_snapshot=False,
    )
    bound_run = get_bound_particles(sim_run, mass_run)  # Identify the bound remnant at each snapshot.
    traj_run = trajectories(bound_run, sim_run)  # Convert bound summaries into time series.
    t_myr = traj_run["time"] * 1000.0
    M_over_M0 = traj_run["mass"] / traj_run["mass"][0]  # Normalized curve used in every panel.
    table = np.column_stack((t_myr, M_over_M0))

    if use_cache:
        # Save the two-column curve and then enrich the sidecar with the full cache payload.
        write_binary_table(
            cache_path,
            table,
            columns=["elapsed_time", "bound_mass_fraction"],
            units={"elapsed_time": "Myr"},
            description="Cached NGC6569 mass-loss curve for the Dev3 Step 2 convergence sweep.",
        )
        meta = json.loads(cache_path.with_suffix(".json").read_text())
        meta["cache_key"] = binary_cache_key(payload)
        meta["cache_payload"] = payload
        write_manifest(cache_path.with_suffix(".json"), meta)

    return t_myr, M_over_M0


In [ ]:
## Step 2 driver: one-parameter-at-a-time convergence panels
# Flip this to True when you want to launch the full cached sweep.
RUN_NGC6569_STEP2 = False

if RUN_NGC6569_STEP2:
    import matplotlib.cm as cm
    from matplotlib.colors import Normalize

    fid = {  # Dev3 fiducial values that every one-parameter sweep brackets.
        "W0": float(W0_value),
        "scale_radius": float(r_scale),
        "kmax": int(kmax),
        "eps_pc": float(eps * 1000.0),
        "m_star": float(m / Nbody),
    }

    W_values = [2, 4, 6, 7, 8, 10, 12, 14]  # Include the fiducial W0=7 explicitly.
    kmax_values = [14, 15, 16, 17, 18]  # Bracket the Dev3 timestep kmax=16.
    eps_values = [2**-6, 2**-5, 2**-4, 2**-3, 2**-2]  # 0.015625 through 0.25 pc.
    m_star_values = [fid["m_star"] * factor for factor in [0.25, 0.5, 1.0, 2.0, 4.0]]  # Bracket Nbody.

    _king_trunc_cache = {}  # Avoid recomputing King truncation radii for repeated W values.

    def _king_trunc(W):
        if W not in _king_trunc_cache:
            _king_trunc_cache[W] = king_trunc_ratio(W)
        return _king_trunc_cache[W]

    def _tidal_matched_scale_radius(W):
        if king_trunc_ratio is None:
            return fid["scale_radius"]
        fid_tidal_radius = fid["scale_radius"] * _king_trunc(fid["W0"])  # Hold r_t fixed across W0.
        return fid_tidal_radius / _king_trunc(W)

    results = {"W": {}, "tau": {}, "eps": {}, "m": {}}  # Each entry stores (t_myr, M_over_M0).
    w_scale_radii = {}

    for W in W_values:
        scale_radius_W = _tidal_matched_scale_radius(W)  # W panel changes W0 but keeps tidal size matched.
        w_scale_radii[W] = scale_radius_W
        results["W"][W] = run_ngc6569_mass_loss(
            W, scale_radius_W, fid["kmax"], fid["eps_pc"], fid["m_star"]
        )

    for k in kmax_values:
        results["tau"][k] = run_ngc6569_mass_loss(
            fid["W0"], fid["scale_radius"], k, fid["eps_pc"], fid["m_star"]
        )

    for e in eps_values:
        results["eps"][e] = run_ngc6569_mass_loss(
            fid["W0"], fid["scale_radius"], fid["kmax"], e, fid["m_star"]
        )

    for ms in m_star_values:
        results["m"][ms] = run_ngc6569_mass_loss(
            fid["W0"], fid["scale_radius"], fid["kmax"], fid["eps_pc"], ms
        )

    def _colors(values, fiducial):
        # Fiducial curve is black; other values map smoothly across coolwarm.
        vals = sorted(values)
        norm = Normalize(vmin=min(vals), vmax=max(vals))
        return {v: ("black" if np.isclose(v, fiducial) else cm.coolwarm(norm(v))) for v in vals}

    def _curve_lw(color):
        return 2.4 if color == "black" else 1.4  # Make the fiducial curve visually dominant.

    fig, axes = plt.subplots(4, 1, figsize=(7, 14), sharex=True)  # One row per varied parameter.
    fig.suptitle("NGC6569 Dev3 one-parameter convergence sweep", size=16)

    ax = axes[0]
    cols = _colors(results["W"], fid["W0"])
    for W in sorted(results["W"]):
        t, Mfrac = results["W"][W]
        label = f"W0 = {W:g}, r_s = {w_scale_radii[W] * 1000.0:.3g} pc"
        ax.plot(t, Mfrac, color=cols[W], lw=_curve_lw(cols[W]), label=label)
    ax.set_title("W0 sweep, King tidal radius matched")

    ax = axes[1]
    cols = _colors(results["tau"], fid["kmax"])
    for k in sorted(results["tau"]):
        t, Mfrac = results["tau"][k]
        ax.plot(t, Mfrac, color=cols[k], lw=_curve_lw(cols[k]), label=f"kmax = {k}")
    ax.set_title("Timestep sweep")

    ax = axes[2]
    cols = _colors(results["eps"], fid["eps_pc"])
    for e in sorted(results["eps"]):
        t, Mfrac = results["eps"][e]
        ax.plot(t, Mfrac, color=cols[e], lw=_curve_lw(cols[e]), label=f"eps = {e:g} pc")
    ax.set_title("Softening sweep")

    ax = axes[3]
    cols = _colors(results["m"], fid["m_star"])
    for ms in sorted(results["m"]):
        t, Mfrac = results["m"][ms]
        ax.plot(t, Mfrac, color=cols[ms], lw=_curve_lw(cols[ms]), label=f"m_star = {ms:g} Msun")
    ax.set_title("Particle-mass sweep")

    for ax in axes:
        ax.set_xlim(0, abs(tfin.to(u.Myr).value))
        ax.set_ylim(0.7, 1.0)  # Zoom in on convergence instead of the crash-to-zero regime.
        ax.set_ylabel(r"$M(t)/M_0$", size=14)
        ax.grid(alpha=0.25)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9)

    axes[-1].set_xlabel(r"$t$ [Myr]", size=14)
    fig.tight_layout()
    fig.savefig(STEP2_OUT / "ngc6569_step2_recentered_massloss.pdf", bbox_inches="tight")
else:
    print("Set RUN_NGC6569_STEP2 = True to run the optional cached convergence sweep.")
